# Spatial Data Clustering with DBSCAN

**Estimated Time:** 45-60 minutes  
**Difficulty:** Intermediate

## Learning Objectives

By the end of this notebook, you will be able to:
- Apply DBSCAN to real-world spatial/geographic data
- Understand domain-specific parameter selection for spatial clustering
- Process GPS trajectory data for clustering analysis
- Interpret spatial clustering results in geographic context
- Benchmark DBSCAN performance on spatial datasets
- Handle coordinate systems and distance metrics for geographic data

## Prerequisites

- Completion of `01_dbscan_basics.ipynb`
- Completion of `06_parameter_tuning.ipynb` (recommended)
- Basic understanding of geographic coordinates (latitude/longitude)
- Familiarity with NumPy and Matplotlib

## Paper References

This notebook demonstrates concepts from:
- **Section 1** (p. 226): Motivation - spatial databases
- **Section 5.1** (p. 229): Parameter determination for spatial data
- **Section 6** (p. 230): Performance evaluation on spatial datasets

**Full Reference:**  
Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. In *KDD* (Vol. 96, No. 34, pp. 226-231).

---

## Table of Contents

1. [Introduction to Spatial Clustering](#introduction)
2. [Setup and Imports](#setup)
3. [Loading Real GPS Trajectory Data](#loading-data)
4. [Data Preprocessing](#preprocessing)
5. [Domain-Specific Parameter Selection](#parameter-selection)
6. [Applying DBSCAN to Spatial Data](#clustering)
7. [Interpreting Results](#interpretation)
8. [Performance Benchmarks](#performance)
9. [Advanced: Distance Metrics for Geographic Data](#distance-metrics)
10. [Exercises](#exercises)
11. [Summary](#summary)
12. [Next Steps](#next-steps)

---

## 1. Introduction to Spatial Clustering <a id='introduction'></a>

### Why Spatial Clustering?

Spatial data clustering is one of the primary motivations for DBSCAN [Paper §1, p. 226]. Geographic and spatial datasets have unique characteristics:

- **Irregular shapes**: Geographic clusters rarely form perfect circles
- **Varying density**: Urban areas vs. rural areas have different point densities
- **Noise and outliers**: GPS errors, isolated locations
- **Large scale**: Millions of points in real-world applications

### Real-World Applications

1. **Transportation**: Identifying traffic hotspots, route clustering
2. **Urban Planning**: Finding activity centers, commercial districts
3. **Ecology**: Animal movement patterns, habitat identification
4. **Epidemiology**: Disease outbreak clustering
5. **Retail**: Customer location analysis, store placement

### Challenges with Spatial Data

- **Coordinate systems**: Latitude/longitude vs. projected coordinates
- **Distance metrics**: Euclidean vs. Haversine (great-circle) distance
- **Scale**: Choosing appropriate epsilon in meters/kilometers
- **Density variations**: Different densities across geographic regions

---

## 2. Setup and Imports <a id='setup'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.metrics import silhouette_score
import sys
sys.path.append('..')

from src.dbscan_from_scratch import DBSCAN
from src.visualization import DBSCANVisualizer
from src.parameter_tuning import ParameterSelector

# Set random seed for reproducibility
np.random.seed(42)

# Initialize visualizer and parameter selector
viz = DBSCANVisualizer()
param_selector = ParameterSelector()

print("✓ Setup complete!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---

## 3. Loading Real GPS Trajectory Data <a id='loading-data'></a>

We'll work with simulated GPS trajectory data representing vehicle movements in an urban area.

### Generate Realistic GPS Trajectory Data

Since we're creating a learning resource, we'll generate realistic GPS data with known patterns:

In [ ]:
def generate_gps_trajectory_data(n_trajectories=50, points_per_trajectory=20, random_state=42):
    """
    Generate realistic GPS trajectory data simulating vehicle movements.
    
    Creates multiple hotspots (downtown, shopping centers, residential areas)
    with trajectories connecting them.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: trajectory_id, latitude, longitude, timestamp
    """
    np.random.seed(random_state)
    
    # Define hotspot centers (lat, lon) - simulating a city
    hotspots = [
        (40.7580, -73.9855),  # Downtown (Times Square area)
        (40.7489, -73.9680),  # Midtown East
        (40.7614, -73.9776),  # Central Park South
        (40.7505, -73.9934),  # Chelsea
        (40.7282, -73.9942),  # Greenwich Village
    ]
    
    trajectories = []
    
    for traj_id in range(n_trajectories):
        # Each trajectory visits 2-3 hotspots
        n_hotspots = np.random.randint(2, 4)
        visited_hotspots = np.random.choice(len(hotspots), n_hotspots, replace=False)
        
        points_per_hotspot = points_per_trajectory // n_hotspots
        
        for hotspot_idx in visited_hotspots:
            center_lat, center_lon = hotspots[hotspot_idx]
            
            # Generate points around hotspot with GPS noise
            lats = np.random.normal(center_lat, 0.002, points_per_hotspot)
            lons = np.random.normal(center_lon, 0.002, points_per_hotspot)
            
            for i, (lat, lon) in enumerate(zip(lats, lons)):
                trajectories.append({
                    'trajectory_id': traj_id,
                    'latitude': lat,
                    'longitude': lon,
                    'timestamp': traj_id * points_per_trajectory + i
                })
    
    # Add some noise points (GPS errors, outliers)
    n_noise = int(len(trajectories) * 0.05)
    for i in range(n_noise):
        trajectories.append({
            'trajectory_id': -1,  # Noise trajectory
            'latitude': np.random.uniform(40.70, 40.80),
            'longitude': np.random.uniform(-74.02, -73.95),
            'timestamp': -1
        })
    
    df = pd.DataFrame(trajectories)
    return df

# Generate data
gps_data = generate_gps_trajectory_data(n_trajectories=50, points_per_trajectory=20)

print(f"\n📍 GPS Trajectory Data Generated:")
print(f"  • Total points: {len(gps_data)}")
print(f"  • Unique trajectories: {gps_data['trajectory_id'].nunique() - 1}")  # -1 for noise
print(f"  • Latitude range: [{gps_data['latitude'].min():.4f}, {gps_data['latitude'].max():.4f}]")
print(f"  • Longitude range: [{gps_data['longitude'].min():.4f}, {gps_data['longitude'].max():.4f}]")
print(f"\nFirst few rows:")
print(gps_data.head(10))

### Visualize Raw GPS Data

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(gps_data['longitude'], gps_data['latitude'], 
           c='gray', alpha=0.5, s=20, edgecolors='black', linewidths=0.5)
plt.title('Raw GPS Trajectory Data (Simulated Urban Area)', fontsize=14, fontweight='bold')
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n🗺️ Visualization Notes:")
print("  • Each point represents a GPS location")
print("  • Visual clusters suggest activity hotspots")
print("  • Scattered points may be GPS errors or isolated locations")

---

## 4. Data Preprocessing <a id='preprocessing'></a>

### Extract Coordinate Features

For DBSCAN, we need to prepare our feature matrix X containing latitude and longitude coordinates.

In [ ]:
# Extract coordinates as feature matrix
X_gps = gps_data[['latitude', 'longitude']].values

print(f"Feature Matrix Shape: {X_gps.shape}")
print(f"  • {X_gps.shape[0]} points")
print(f"  • {X_gps.shape[1]} features (latitude, longitude)")
print(f"\nSample coordinates:")
print(X_gps[:5])

### Understanding Coordinate Scale

**Important**: For geographic data, we need to understand the scale:

- **1 degree latitude** ≈ 111 km (constant)
- **1 degree longitude** ≈ 111 km × cos(latitude) (varies by latitude)
- At latitude 40°: 1 degree ≈ 85 km

For our data:
- **0.001 degrees** ≈ 111 meters (latitude) or ≈ 85 meters (longitude at 40°)
- **0.01 degrees** ≈ 1.1 km (latitude) or ≈ 850 meters (longitude at 40°)

In [ ]:
# Calculate data extent
lat_range = X_gps[:, 0].max() - X_gps[:, 0].min()
lon_range = X_gps[:, 1].max() - X_gps[:, 1].min()

# Approximate distances
lat_distance_km = lat_range * 111
lon_distance_km = lon_range * 85  # At latitude ~40°

print(f"\n📏 Data Extent:")
print(f"  • Latitude range: {lat_range:.4f}° ≈ {lat_distance_km:.2f} km")
print(f"  • Longitude range: {lon_range:.4f}° ≈ {lon_distance_km:.2f} km")
print(f"\n💡 Parameter Selection Hint:")
print(f"  • For 200m radius: eps ≈ 0.002 degrees")
print(f"  • For 500m radius: eps ≈ 0.005 degrees")
print(f"  • For 1km radius: eps ≈ 0.010 degrees")

---

## 5. Domain-Specific Parameter Selection <a id='parameter-selection'></a>

### Strategy for Spatial Data [Paper §5.1, p. 229]

For spatial/geographic data, parameter selection should consider:

1. **Domain Knowledge**:
   - What distance defines "nearby" in your application?
   - Urban hotspot: 200-500 meters
   - Neighborhood: 1-2 kilometers
   - City district: 5-10 kilometers

2. **Data Characteristics**:
   - Point density
   - Expected cluster sizes
   - GPS accuracy (typically 5-10 meters)

3. **MinPts Heuristic** [Paper §5.1]:
   - For 2D spatial data: MinPts = 4-6 is typical
   - Higher MinPts for noisy GPS data
   - Lower MinPts for sparse rural data

### K-Distance Graph for Epsilon Selection

In [ ]:
# Compute k-distances for parameter selection
k = 5  # Typical MinPts for 2D spatial data
k_distances = param_selector.compute_k_distances(X_gps, k=k)

# Find elbow point
elbow_idx, suggested_eps = param_selector.find_elbow_point(k_distances)

# Visualize k-distance graph
viz.plot_k_distance_graph(X_gps, k=k, show_elbow=True)
plt.show()

print(f"\n📊 K-Distance Analysis (k={k}):")
print(f"  • Suggested epsilon: {suggested_eps:.6f} degrees")
print(f"  • Approximate distance: {suggested_eps * 111:.0f} meters (latitude)")
print(f"  • Approximate distance: {suggested_eps * 85:.0f} meters (longitude at 40°)")
print(f"\n💡 Interpretation:")
print(f"  • Points within ~{suggested_eps * 100:.0f}m are considered neighbors")
print(f"  • This captures activity hotspots at street/block level")

### Automatic Parameter Suggestion

In [ ]:
# Get automatic parameter suggestions
suggested_eps_auto, suggested_minpts_auto = param_selector.suggest_parameters(X_gps)

print(f"\n🤖 Automatic Parameter Suggestions:")
print(f"  • Epsilon: {suggested_eps_auto:.6f} degrees")
print(f"  • MinPts: {suggested_minpts_auto}")
print(f"  • Approximate radius: {suggested_eps_auto * 100:.0f} meters")
print(f"\n📝 Recommendation:")
print(f"  • Use these as starting points")
print(f"  • Adjust based on domain knowledge and visual inspection")
print(f"  • For urban hotspots, eps=0.003-0.005 (300-500m) works well")

---

## 6. Applying DBSCAN to Spatial Data <a id='clustering'></a>

### Complete Workflow: Data → Parameters → Clustering → Results

In [ ]:
# Set parameters based on domain knowledge and k-distance analysis
# We want to identify activity hotspots at ~300-400 meter radius
eps_spatial = 0.004  # ~400 meters
min_pts_spatial = 5  # Typical for 2D spatial data

print(f"\n⚙️ DBSCAN Parameters:")
print(f"  • Epsilon: {eps_spatial} degrees (~{eps_spatial * 100:.0f}m)")
print(f"  • MinPts: {min_pts_spatial}")
print(f"  • Interpretation: Points within ~400m with ≥5 neighbors form clusters")

# Apply DBSCAN
print(f"\n🔄 Running DBSCAN...")
start_time = time.time()

dbscan_spatial = DBSCAN(eps=eps_spatial, min_pts=min_pts_spatial)
labels_spatial = dbscan_spatial.fit_predict(X_gps)

elapsed_time = time.time() - start_time

# Analyze results
n_clusters = len(set(labels_spatial)) - (1 if -1 in labels_spatial else 0)
n_noise = list(labels_spatial).count(-1)
n_core = len(dbscan_spatial.get_core_points())

print(f"\n✅ DBSCAN Complete! (Time: {elapsed_time:.3f}s)")
print(f"\n📊 Clustering Results:")
print(f"  • Clusters found: {n_clusters}")
print(f"  • Core points: {n_core} ({100*n_core/len(X_gps):.1f}%)")
print(f"  • Noise points: {n_noise} ({100*n_noise/len(X_gps):.1f}%)")
print(f"  • Total points: {len(X_gps)}")

# Calculate cluster sizes
if n_clusters > 0:
    cluster_sizes = [np.sum(labels_spatial == i) for i in range(n_clusters)]
    print(f"\n📍 Cluster Sizes:")
    for i, size in enumerate(cluster_sizes):
        print(f"  • Cluster {i}: {size} points")

### Visualize Clustering Results

In [ ]:
# Create visualization
plt.figure(figsize=(14, 10))

# Plot clusters
unique_labels = set(labels_spatial)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    if label == -1:
        # Noise points
        mask = labels_spatial == label
        plt.scatter(X_gps[mask, 1], X_gps[mask, 0], 
                   c='black', marker='x', s=50, alpha=0.5, label='Noise')
    else:
        # Cluster points
        mask = labels_spatial == label
        plt.scatter(X_gps[mask, 1], X_gps[mask, 0], 
                   c=[color], s=80, alpha=0.7, edgecolors='black', linewidths=0.5,
                   label=f'Hotspot {label}')

plt.title(f'Spatial Clustering Results: Activity Hotspots\n(ε={eps_spatial}° ≈ {eps_spatial*100:.0f}m, MinPts={min_pts_spatial})', 
         fontsize=14, fontweight='bold')
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n🗺️ Map Interpretation:")
print(f"  • Each color represents a distinct activity hotspot")
print(f"  • Hotspots are areas with high GPS point density")
print(f"  • Black 'x' markers are isolated/error points")
print(f"  • Cluster shapes reflect actual geographic patterns")

---

## 7. Interpreting Results <a id='interpretation'></a>

### Geographic Context

Let's analyze what each cluster represents in geographic terms:

In [ ]:
# Calculate cluster centroids and characteristics
print("\n📍 Cluster Analysis:\n")

for cluster_id in range(n_clusters):
    mask = labels_spatial == cluster_id
    cluster_points = X_gps[mask]
    
    # Calculate centroid
    centroid_lat = cluster_points[:, 0].mean()
    centroid_lon = cluster_points[:, 1].mean()
    
    # Calculate spread (standard deviation)
    spread_lat = cluster_points[:, 0].std() * 111 * 1000  # meters
    spread_lon = cluster_points[:, 1].std() * 85 * 1000   # meters
    
    # Count core vs border points
    core_mask = np.isin(np.where(mask)[0], dbscan_spatial.core_sample_indices_)
    n_core_cluster = np.sum(core_mask)
    n_border_cluster = np.sum(mask) - n_core_cluster
    
    print(f"Hotspot {cluster_id}:")
    print(f"  • Location: ({centroid_lat:.4f}°, {centroid_lon:.4f}°)")
    print(f"  • Size: {np.sum(mask)} points")
    print(f"  • Composition: {n_core_cluster} core, {n_border_cluster} border")
    print(f"  • Spread: ±{spread_lat:.0f}m (N-S), ±{spread_lon:.0f}m (E-W)")
    print(f"  • Density: {np.sum(mask) / (spread_lat * spread_lon / 1e6):.1f} points/km²")
    print()

### Point Type Distribution

In [ ]:
# Visualize point types
viz.plot_point_types(
    X_gps, 
    labels_spatial, 
    dbscan_spatial.core_sample_indices_, 
    eps=eps_spatial,
    title=f"Point Type Classification for Spatial Data (ε={eps_spatial}°, MinPts={min_pts_spatial})"
)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

# Calculate statistics
core_mask = np.zeros(len(X_gps), dtype=bool)
core_mask[dbscan_spatial.core_sample_indices_] = True
noise_mask = labels_spatial == -1
border_mask = ~core_mask & ~noise_mask

print("\n📊 Point Type Distribution:")
print(f"  • Core points: {np.sum(core_mask)} ({100*np.sum(core_mask)/len(X_gps):.1f}%)")
print(f"    → Centers of activity hotspots")
print(f"  • Border points: {np.sum(border_mask)} ({100*np.sum(border_mask)/len(X_gps):.1f}%)")
print(f"    → Edges of hotspots, less dense areas")
print(f"  • Noise points: {np.sum(noise_mask)} ({100*np.sum(noise_mask)/len(X_gps):.1f}%)")
print(f"    → GPS errors, isolated locations, transit points")

---

## 8. Performance Benchmarks <a id='performance'></a>

### Benchmark on Varying Dataset Sizes [Paper §6, p. 230]

Let's measure DBSCAN performance on spatial datasets of different sizes:

In [ ]:
# Benchmark different dataset sizes
sizes = [100, 500, 1000, 2000, 5000]
times = []
n_clusters_list = []

print("\n⏱️ Performance Benchmark:\n")
print(f"{'Size':<10} {'Time (s)':<12} {'Clusters':<10} {'Points/sec':<12}")
print("-" * 50)

for size in sizes:
    # Generate dataset
    gps_test = generate_gps_trajectory_data(
        n_trajectories=size//20, 
        points_per_trajectory=20,
        random_state=42
    )
    X_test = gps_test[['latitude', 'longitude']].values[:size]
    
    # Run DBSCAN and measure time
    start = time.time()
    dbscan_test = DBSCAN(eps=eps_spatial, min_pts=min_pts_spatial)
    labels_test = dbscan_test.fit_predict(X_test)
    elapsed = time.time() - start
    
    n_clusters_test = len(set(labels_test)) - (1 if -1 in labels_test else 0)
    throughput = size / elapsed
    
    times.append(elapsed)
    n_clusters_list.append(n_clusters_test)
    
    print(f"{size:<10} {elapsed:<12.4f} {n_clusters_test:<10} {throughput:<12.0f}")

print("\n📈 Performance Characteristics:")
print(f"  • Algorithm: O(n²) naive implementation")
print(f"  • Scalability: {times[-1]/times[0]:.1f}x slower for {sizes[-1]/sizes[0]:.0f}x more data")
print(f"  • Note: Spatial indexing (KD-tree) can reduce to O(n log n)")

### Visualize Performance Scaling

In [ ]:
# Plot performance results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Runtime vs dataset size
ax1.plot(sizes, times, 'o-', linewidth=2, markersize=8, color='#1f77b4')
ax1.set_xlabel('Dataset Size (points)', fontsize=11)
ax1.set_ylabel('Runtime (seconds)', fontsize=11)
ax1.set_title('DBSCAN Runtime vs Dataset Size', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Add O(n²) reference curve
theoretical = np.array(sizes)**2 / (sizes[0]**2) * times[0]
ax1.plot(sizes, theoretical, '--', linewidth=2, color='red', alpha=0.5, label='O(n²) theoretical')
ax1.legend()

# Throughput
throughput = np.array(sizes) / np.array(times)
ax2.plot(sizes, throughput, 'o-', linewidth=2, markersize=8, color='#2ca02c')
ax2.set_xlabel('Dataset Size (points)', fontsize=11)
ax2.set_ylabel('Throughput (points/second)', fontsize=11)
ax2.set_title('DBSCAN Throughput', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Performance Insights:")
print(f"  • Runtime grows quadratically with dataset size (O(n²))")
print(f"  • For {sizes[-1]} points: {times[-1]:.2f}s")
print(f"  • Throughput decreases as dataset grows")
print(f"  • Optimization: Use spatial indexing for large datasets (>10k points)")

### Clustering Quality Metrics

In [ ]:
# Calculate silhouette score (if we have clusters)
if n_clusters >= 2:
    # Compute only on non-noise points
    non_noise_mask = labels_spatial != -1
    if np.sum(non_noise_mask) >= 2:
        silhouette_avg = silhouette_score(X_gps[non_noise_mask], labels_spatial[non_noise_mask])
        
        print(f"\n📊 Clustering Quality Metrics:")
        print(f"  • Silhouette Score: {silhouette_avg:.3f}")
        print(f"    → Range: [-1, 1], higher is better")
        print(f"    → {silhouette_avg:.3f} indicates {'excellent' if silhouette_avg > 0.7 else 'good' if silhouette_avg > 0.5 else 'fair' if silhouette_avg > 0.3 else 'poor'} clustering")
        print(f"  • Number of clusters: {n_clusters}")
        print(f"  • Noise ratio: {100*n_noise/len(X_gps):.1f}%")
else:
    print("\n⚠️ Not enough clusters for silhouette score calculation")

---

## 9. Advanced: Distance Metrics for Geographic Data <a id='distance-metrics'></a>

### Euclidean vs Haversine Distance

For small geographic areas (< 100km), Euclidean distance on lat/lon coordinates is acceptable. For larger areas or higher precision, use Haversine distance (great-circle distance).

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two points on Earth.
    
    Parameters
    ----------
    lat1, lon1, lat2, lon2 : float
        Coordinates in degrees
    
    Returns
    -------
    float
        Distance in kilometers
    """
    # Convert to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Earth radius in kilometers
    r = 6371
    
    return r * c

# Compare distances
p1 = X_gps[0]
p2 = X_gps[10]

# Euclidean distance (in degrees)
euclidean_deg = np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)
euclidean_km = euclidean_deg * 100  # Rough approximation

# Haversine distance
haversine_km = haversine_distance(p1[0], p1[1], p2[0], p2[1])

print(f"\n📏 Distance Comparison:")
print(f"  Point 1: ({p1[0]:.4f}°, {p1[1]:.4f}°)")
print(f"  Point 2: ({p2[0]:.4f}°, {p2[1]:.4f}°)")
print(f"\n  Euclidean: {euclidean_deg:.6f}° ≈ {euclidean_km:.3f} km (approximation)")
print(f"  Haversine: {haversine_km:.3f} km (exact great-circle)")
print(f"  Difference: {abs(euclidean_km - haversine_km):.3f} km ({100*abs(euclidean_km - haversine_km)/haversine_km:.1f}%)")
print(f"\n💡 When to use Haversine:")
print(f"  • Large geographic areas (> 100km)")
print(f"  • High precision requirements")
print(f"  • Near poles (where longitude distortion is high)")
print(f"\n💡 When Euclidean is OK:")
print(f"  • Small areas (< 100km)")
print(f"  • Mid-latitudes (20°-60°)")
print(f"  • Computational efficiency is important")

---

## 10. Exercises <a id='exercises'></a>

### Exercise 1: Parameter Tuning for Different Scales (Intermediate)

**Task**: The current parameters identify hotspots at ~400m radius. Adjust parameters to identify:
1. Fine-grained hotspots (~200m radius)
2. Neighborhood-level clusters (~1km radius)

Compare the number of clusters found at each scale.

In [ ]:
# Exercise 1: Your code here
# TODO: Try eps ≈ 0.002 for 200m and eps ≈ 0.010 for 1km
# eps_fine = ???
# eps_neighborhood = ???

<details>
<summary><b>Click to reveal Exercise 1 solution</b></summary>

**Solution**:

In [ ]:
# Exercise 1 Solution
scales = [
    (0.002, "Fine-grained (~200m)"),
    (0.004, "Current (~400m)"),
    (0.010, "Neighborhood (~1km)")
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (eps, label) in zip(axes, scales):
    dbscan_ex = DBSCAN(eps=eps, min_pts=5)
    labels_ex = dbscan_ex.fit_predict(X_gps)
    
    n_clusters_ex = len(set(labels_ex)) - (1 if -1 in labels_ex else 0)
    n_noise_ex = list(labels_ex).count(-1)
    
    # Plot
    unique_labels = set(labels_ex)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for lbl, color in zip(unique_labels, colors):
        if lbl == -1:
            mask = labels_ex == lbl
            ax.scatter(X_gps[mask, 1], X_gps[mask, 0], c='black', marker='x', s=30, alpha=0.5)
        else:
            mask = labels_ex == lbl
            ax.scatter(X_gps[mask, 1], X_gps[mask, 0], c=[color], s=50, alpha=0.7)
    
    ax.set_title(f'{label}\n{n_clusters_ex} clusters, {n_noise_ex} noise', fontsize=11, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Scale Comparison:")
print("  • Fine-grained (200m): More clusters, captures local variations")
print("  • Current (400m): Balanced, good for activity hotspots")
print("  • Neighborhood (1km): Fewer clusters, broader regions")
print("\n💡 Insight: Scale selection depends on application needs!")

</details>

### Exercise 2: Analyzing Cluster Density (Advanced)

**Task**: For each cluster, calculate:
1. Average distance to cluster centroid
2. Maximum distance to cluster centroid
3. Density (points per square kilometer)

Which cluster is the most dense? Which is the most spread out?

In [ ]:
# Exercise 2: Your code here
# TODO: Calculate cluster statistics

<details>
<summary><b>Click to reveal Exercise 2 solution</b></summary>

**Solution**:

In [ ]:
# Exercise 2 Solution
print("\n📊 Detailed Cluster Analysis:\n")

cluster_stats = []

for cluster_id in range(n_clusters):
    mask = labels_spatial == cluster_id
    cluster_points = X_gps[mask]
    
    # Centroid
    centroid = cluster_points.mean(axis=0)
    
    # Distances to centroid (in meters)
    distances = np.sqrt(
        ((cluster_points[:, 0] - centroid[0]) * 111000)**2 + 
        ((cluster_points[:, 1] - centroid[1]) * 85000)**2
    )
    
    avg_dist = distances.mean()
    max_dist = distances.max()
    
    # Approximate area (circle with radius = max_dist)
    area_km2 = np.pi * (max_dist / 1000)**2
    density = len(cluster_points) / area_km2 if area_km2 > 0 else 0
    
    cluster_stats.append({
        'id': cluster_id,
        'size': len(cluster_points),
        'avg_dist': avg_dist,
        'max_dist': max_dist,
        'density': density
    })
    
    print(f"Cluster {cluster_id}:")
    print(f"  • Size: {len(cluster_points)} points")
    print(f"  • Avg distance to centroid: {avg_dist:.0f}m")
    print(f"  • Max distance to centroid: {max_dist:.0f}m")
    print(f"  • Approximate area: {area_km2:.3f} km²")
    print(f"  • Density: {density:.1f} points/km²")
    print()

# Find most/least dense
densest = max(cluster_stats, key=lambda x: x['density'])
most_spread = max(cluster_stats, key=lambda x: x['max_dist'])

print(f"\n🏆 Analysis Results:")
print(f"  • Most dense: Cluster {densest['id']} ({densest['density']:.1f} points/km²)")
print(f"  • Most spread out: Cluster {most_spread['id']} (radius {most_spread['max_dist']:.0f}m)")

</details>

---

## 11. Summary <a id='summary'></a>

### Key Takeaways

1. **Spatial Data Characteristics**:
   - Geographic data has irregular shapes and varying densities
   - DBSCAN is well-suited for spatial clustering [Paper §1, p. 226]
   - Coordinate systems and distance metrics matter

2. **Complete Workflow**:
   - Load and preprocess GPS data
   - Use k-distance graph for parameter selection [Paper §5.1]
   - Apply DBSCAN with domain-appropriate parameters
   - Interpret results in geographic context
   - Benchmark performance

3. **Domain-Specific Parameter Selection**:
   - **Epsilon**: Choose based on meaningful distance (200m-1km for urban)
   - **MinPts**: 4-6 typical for 2D spatial data
   - Use k-distance graph as starting point
   - Adjust based on domain knowledge and visual inspection

4. **Performance Considerations** [Paper §6, p. 230]:
   - Naive implementation: O(n²) time complexity
   - Spatial indexing can reduce to O(n log n)
   - Trade-off between accuracy and speed

5. **Practical Applications**:
   - Activity hotspot identification
   - Traffic pattern analysis
   - Urban planning and retail location analysis
   - Ecological movement patterns

### What We Learned

- ✅ How to process real GPS trajectory data
- ✅ Domain-specific parameter selection strategies
- ✅ Complete clustering workflow from data to interpretation
- ✅ Performance benchmarking on spatial datasets
- ✅ Distance metric considerations (Euclidean vs Haversine)
- ✅ Geographic interpretation of clustering results

---

## 12. Next Steps <a id='next-steps'></a>

### Recommended Next Topics

1. **Anomaly Detection** (`09_anomaly_detection.ipynb`):
   - Use DBSCAN for outlier detection
   - Identify GPS errors and unusual patterns

2. **Performance Optimization** (`10_performance_analysis.ipynb`):
   - Spatial indexing (KD-tree, R-tree)
   - Scaling to large datasets (>100k points)

3. **Advanced Topics** (`11_advanced_topics.ipynb`):
   - OPTICS for varying density
   - HDBSCAN for hierarchical clustering
   - Custom distance metrics

### Further Reading

- **Paper Section 6** (p. 230): Performance evaluation on spatial databases
- **docs/07_applications.md**: More real-world use cases
- **docs/08_performance_optimization.md**: Scaling strategies

### Practice Suggestions

1. Apply DBSCAN to your own GPS data
2. Experiment with different distance metrics
3. Try clustering at multiple scales
4. Compare with other clustering algorithms (K-Means, Hierarchical)
5. Implement Haversine distance for large-scale geographic data

---

**Congratulations!** You've completed the spatial clustering notebook. You now understand how to apply DBSCAN to real-world geographic data with domain-appropriate parameter selection and performance considerations.